In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

/Users/cinar/.pyenv/versions/3.13.14/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
documents = [
    "RAG, retrieval ve generation süreçlerini birleştirerek LLM doğruluğunu artırır.",
    "Vector database, metinlerin embedding temsillerini saklamak için kullanılır.",
    "Embedding, metinleri yüksek boyutlu sayısal vektörlere dönüştürür.",
    "Similarity search, en alakalı doküman parçalarını bulmak için kullanılır.",
    "Hybrid search, semantic search ile keyword search yöntemlerini birleştirir.",
    "Re-ranking, ilk bulunan sonuçları yeniden değerlendirerek daha doğru sıralama yapar."
]

In [3]:
model=SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7425.46it/s]


In [4]:
doc_embeddings=model.encode(documents)

In [5]:
query="Hybrid search neden kullanılır"

In [6]:
query_embedding=model.encode(query)

In [7]:
semantic_scores=np.dot(doc_embeddings,query_embedding.T).flatten()

In [10]:
vectorizer=TfidfVectorizer()
tfidf_matrix=vectorizer.fit_transform(documents)

In [11]:
keyword_scores=vectorizer.transform([query])

In [12]:
keyword_scores=np.dot(tfidf_matrix.toarray(),keyword_scores.toarray().T).flatten()

In [13]:
keyword_scores

array([0.        , 0.16576831, 0.        , 0.31017182, 0.56758489,
       0.        ])

In [14]:
alpha=0.6

hybrid_scores=(alpha * semantic_scores + (1 - alpha) * keyword_scores)



In [15]:
best_index=np.argmax(hybrid_scores)

In [16]:
best_index

np.int64(4)

In [17]:
query

'Hybrid search neden kullanılır'

In [18]:
documents[best_index]

'Hybrid search, semantic search ile keyword search yöntemlerini birleştirir.'

In [19]:
top_candidates=np.argsort(hybrid_scores)[::-1][:3]
reranked_scores=[]

In [20]:
for idx in top_candidates:
    score=np.dot(doc_embeddings[idx],query_embedding.T)
    reranked_scores.append(score)
    

In [21]:
final_index=top_candidates[np.argmax(reranked_scores)]

In [23]:
reranked_scores

[np.float32(0.7249191), np.float32(0.61406815), np.float32(0.4150545)]

In [22]:
final_index

np.int64(4)

In [41]:
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
context=documents[final_index]

client = OpenAI(api_key=api_key)



response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "Sadece contexte göre cevap üret cevap ver."
        },
        {
            "role": "user",
            "content": f"""
                    Context: {context}
                    Soru:{query}
                
                """
        }
    ]
)



In [43]:
print(response)

ChatCompletion(id='chatcmpl-DwwAmGNGUsSUtBlcoLKxrIL7yjmBi', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hybrid search, farklı arama yöntemlerini birleştirerek daha kapsamlı ve etkili sonuçlar elde etmek için kullanılır. Semantic search, kullanıcı niyetini ve bağlamı anlamaya odaklanırken, keyword search belirli anahtar kelimelere dayanır. Hybrid search, bu iki yaklaşımın avantajlarını bir araya getirerek, hem anlamı anlama hem de belirli terimleri bulma yeteneğini artırır. Böylece kullanıcılar, daha doğru ve ilgili sonuçlara ulaşabilir, arama deneyimleri iyileşir. Ayrıca, karmaşık sorgular veya daha geniş içerik alanları için daha etkili olabilir.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1782939012, model='gpt-4o-mini-2024-07-18', object='chat.completion', moderation=None, service_tier='default', system_fingerprint='fp_be7b4271a2', usage=CompletionUsage(completi

In [44]:
response.choices[0].message.content

'Hybrid search, farklı arama yöntemlerini birleştirerek daha kapsamlı ve etkili sonuçlar elde etmek için kullanılır. Semantic search, kullanıcı niyetini ve bağlamı anlamaya odaklanırken, keyword search belirli anahtar kelimelere dayanır. Hybrid search, bu iki yaklaşımın avantajlarını bir araya getirerek, hem anlamı anlama hem de belirli terimleri bulma yeteneğini artırır. Böylece kullanıcılar, daha doğru ve ilgili sonuçlara ulaşabilir, arama deneyimleri iyileşir. Ayrıca, karmaşık sorgular veya daha geniş içerik alanları için daha etkili olabilir.'